# Curve Fitting with Gradient Descent

**Topic:** Calculus for Learning Algorithms &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `x_train` `(300,)`, `y_train` `(300,)`: noisy samples of a cubic polynomial on the interval [-2, 2].
- `x_test` `(150,)`: predict `y` at these points. `y_test` is used by the evaluation cell.

## Your task

Fit `y ≈ a·x³ + b·x² + c·x + d` using **your own gradient descent loop** (no `np.polyfit`, no closed-form solve, no sklearn):

1. Build the feature matrix `[x³, x², x, 1]`.
2. Derive the MSE gradient with respect to `(a, b, c, d)` and iterate.
3. Set `predictions` to your fitted values at `x_test`.

Hint: if the loss diverges, your learning rate is too high.

## Evaluation

Test R² must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

rng = np.random.default_rng(17)
coef = np.array([0.6, -0.4, -1.1, 2.0])   # a, b, c, d

def make(n, seed):
    r = np.random.default_rng(seed)
    x = r.uniform(-2, 2, n)
    y = coef[0]*x**3 + coef[1]*x**2 + coef[2]*x + coef[3] + r.normal(0, 0.25, n)
    return x, y

x_train, y_train = make(300, 1)
x_test, y_test = make(150, 2)
print(f"x_train {x_train.shape}, y range [{y_train.min():.2f}, {y_train.max():.2f}]")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_R2 = 0.88

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions, dtype=float).ravel()
_true = np.asarray(y_test, dtype=float).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

_rmse = float(np.sqrt(np.mean((_pred - _true) ** 2)))
_ss_res = float(np.sum((_true - _pred) ** 2))
_ss_tot = float(np.sum((_true - _true.mean()) ** 2))
r2 = 1.0 - _ss_res / _ss_tot
print("=" * 52)
print(f"Test RMSE : {_rmse:.4f}")
print(f"Test R^2  : {r2:.4f}   (target: >= {TARGET_R2:.2f})")
print(f"Worst absolute error: {float(np.max(np.abs(_pred - _true))):.4f}")
print("=" * 52)
if r2 >= TARGET_R2:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

Build the cubic feature matrix once, then iterate plain batch gradient descent on the MSE gradient `2 Fᵀ(Fw - y)/n`. A conservative learning rate with plenty of steps converges to the least-squares fit.

```python
import numpy as np

def features(x):
    return np.column_stack([x**3, x**2, x, np.ones_like(x)])

F = features(x_train)
w = np.zeros(4)
lr = 0.02
for _ in range(20000):
    grad = 2.0 * F.T @ (F @ w - y_train) / len(y_train)
    w -= lr * grad

print("fitted (a, b, c, d):", np.round(w, 3))
predictions = features(x_test) @ w
```
